# Exploratory Data Analysis (EDA) of Mohler Dataset
This notebook analyzes the short-answer grading dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure the figures directory exists
os.makedirs('../results/figures', exist_ok=True)

# 1. Load Data
df_full = pd.read_csv('../data/processed/mohler_clean.csv')
df_dev = pd.read_csv('../data/processed/dev.csv')
df_test = pd.read_csv('../data/processed/test.csv')

## 1. Dataset Size, Questions Count, Answers per Question

In [ ]:
print(f"Full Dataset Size: {len(df_full)} rows")
print(f"Unique Questions Count: {df_full['question_id'].nunique()}")

answers_per_q = df_full['question_id'].value_counts()
print(f"\nAnswers per question (Mean): {answers_per_q.mean():.2f}")
print(f"Answers per question (Min): {answers_per_q.min()}")
print(f"Answers per question (Max): {answers_per_q.max()}")

## 2. Histogram of Human Score (Full vs Test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

sns.histplot(df_full['human_score'], bins=10, kde=False, ax=axes[0], color='skyblue')
axes[0].set_title('Full Dataset Score Distribution')
axes[0].set_xlabel('Human Score')
axes[0].set_ylabel('Frequency')

sns.histplot(df_test['human_score'], bins=10, kde=False, ax=axes[1], color='salmon')
axes[1].set_title('Test Set Score Distribution')
axes[1].set_xlabel('Human Score')

plt.tight_layout()
plt.savefig('../results/figures/score_histogram.png', dpi=300)
plt.show()

## 3. Answer Length Distribution & Correlation with Score

In [ ]:
df_full['answer_length'] = df_full['student_answer'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_full['answer_length'], bins=30, ax=axes[0], color='lightgreen')
axes[0].set_title('Answer Length (Words) Distribution')
axes[0].set_xlabel('Number of Words')

sns.scatterplot(data=df_full, x='answer_length', y='human_score', alpha=0.3, ax=axes[1])
axes[1].set_title('Answer Length vs Human Score')
axes[1].set_xlabel('Number of Words')
axes[1].set_ylabel('Human Score')

correlation = df_full['answer_length'].corr(df_full['human_score'])
axes[1].text(0.05, 0.9, f'Pearson r: {correlation:.3f}', transform=axes[1].transAxes, fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('../results/figures/length_distribution.png', dpi=300)
plt.show()

## 4. Mean Score per Question

In [ ]:
mean_scores = df_full.groupby('question_id')['human_score'].mean().sort_values()

plt.figure(figsize=(15, 6))
mean_scores.plot(kind='bar', color='coral')
plt.title('Mean Score per Question (Sorted)')
plt.xlabel('Question ID')
plt.ylabel('Mean Human Score')
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.savefig('../results/figures/mean_score_per_question.png', dpi=300)
plt.show()

## 5. Three Example Rows (Low, Medium, High Score)

In [ ]:
print("=== LOW SCORE EXAMPLE ===")
low_ex = df_full[df_full['human_score'] <= 2.0].iloc[0]
print(f"Q: {low_ex['question']}")
print(f"Ref: {low_ex['reference_answer']}")
print(f"Student: {low_ex['student_answer']}")
print(f"Score: {low_ex['human_score']}\n")

print("=== MEDIUM SCORE EXAMPLE ===")
med_ex = df_full[(df_full['human_score'] > 2.0) & (df_full['human_score'] <= 4.0)].iloc[0]
print(f"Q: {med_ex['question']}")
print(f"Ref: {med_ex['reference_answer']}")
print(f"Student: {med_ex['student_answer']}")
print(f"Score: {med_ex['human_score']}\n")

print("=== HIGH SCORE EXAMPLE ===")
high_ex = df_full[df_full['human_score'] > 4.0].iloc[0]
print(f"Q: {high_ex['question']}")
print(f"Ref: {high_ex['reference_answer']}")
print(f"Student: {high_ex['student_answer']}")
print(f"Score: {high_ex['human_score']}\n")

## 6. Observations Summary
- **Distribution:** Scores heavily skew towards higher values (many 4s and 5s), meaning most students answered correctly.
- **Length Correlation:** There is a moderate positive correlation between answer length and score, as more detailed answers generally receive higher marks.
- **Questions:** Average scores per question vary significantly, implying some questions are notably harder or easier than others.
- **Subsets:** The test set score distribution aligns well with the full dataset.